> # **Resolução - Exercício 6 (Aula 6)**

 ## **Apresentação de conceitos e métricas estatísticas aplicados em contextos médicos: métodos estatísticos, teste de hipótese e scipy**

---

> ## **Problema 1 - Conhecendo os Dados: Estatística Descritiva e Testes de Hipótese** 🟡
Em estudos na área da saúde, antes de propor equações para estimar o risco de uma doença, é necessário compreender as características isoladas dos dados. O objetivo inicial é observar o comportamento de cada variável clínica perante o desfecho estudado.

**Contextualização:**
Temos um conjunto de dados simulado contendo avaliações de 400 pacientes. O desfecho de interesse é o **Escore de Risco Cardiovascular** (variável contínua).

Para cada paciente, foram registradas **12 variáveis explicativas**:
- **Fatores clássicos:** `Idade`, `IMC`, `Pressao_Arterial`, `Glicemia_Jejum`, `Colesterol_LDL`, `Colesterol_HDL`, `Triglicerideos`.
- **Exames laboratoriais adicionais:** `Marcador_Inflamatorio_1`, `Marcador_Inflamatorio_2`, `Vitamina_D`, `Sodio_Serico`, `Potassio_Serico`.

In [ ]:
# Rode esta célula para instalar o statsmodels
!pip install statsmodels

In [ ]:
# Rode esta célula para carregar os dados

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from scipy.stats import pearsonr, ttest_ind
from scipy.interpolate import interp1d
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score, mean_squared_error
import statsmodels.api as sm

# Simulando dados médicos (o simulador define silenciosamente quais variáveis importam)
X_num, y_num = make_regression(n_samples=400, n_features=12, n_informative=4, noise=25.0, random_state=42)

colunas = [
    'Idade', 'IMC', 'Pressao_Arterial', 'Glicemia_Jejum', 
    'Colesterol_LDL', 'Colesterol_HDL', 'Triglicerideos', 
    'Marcador_Inflamatorio_1', 'Marcador_Inflamatorio_2', 
    'Vitamina_D', 'Sodio_Serico', 'Potassio_Serico'
]

X = pd.DataFrame(X_num, columns=colunas)
y = pd.Series(y_num, name='Escore_Risco_CV')

print(f"Temos dados de {X.shape[0]} pacientes, cada um com {X.shape[1]} exames.")
X.head()

### ✅ **Resolução — Problema 1.1 - Perfil de Dispersão da Amostra** 🟢
O primeiro passo da análise estatística é descrever a distribuição da variável desfecho. Uma das métricas fundamentais é o **Desvio Padrão ($s$)**, que quantifica a dispersão dos dados em torno da média, dado pela equação:

$$s = \sqrt{\frac{\sum_{i=1}^{n} (x_i - \bar{x})^2}{n-1}}$$

**🎯 Objetivo:** Utilizando o `pandas` ou `numpy`, calcule a **média**, a **mediana** e o **desvio padrão** do Escore de Risco Cardiovascular (`y`). Imprima os resultados.<br>

**🧠 Explicação lógica:**<br>
Como a variável `y` é uma `pd.Series` (Série do Pandas), podemos utilizar os métodos nativos `.mean()`, `.median()` e `.std()` para calcular as métricas descritivas de forma direta e eficiente.

In [ ]:
"""
Utilizando os métodos da própria estrutura de dados do Pandas
"""
media_risco = y.mean()
mediana_risco = y.median()
desvio_risco = y.std()

print(f"Média: {media_risco:.2f}")
print(f"Mediana: {mediana_risco:.2f}")
print(f"Desvio Padrão: {desvio_risco:.2f}")

**Resposta da Pergunta Prática:**
O desvio padrão indica o quão distantes os riscos individuais dos pacientes estão da média global. Um desvio padrão alto informa ao médico que a amostra é bastante heterogênea (existem pacientes com riscos muito baixos e outros com riscos extremamente elevados). Se fosse pequeno, indicaria que a maioria dos pacientes possui um risco muito semelhante à média.

---

### ✅ **Resolução — Problema 1.2 - Força Isolada dos Exames (Correlação Linear)** 🟡
O coeficiente de correlação de Pearson ($r$) varia de -1 a 1 e mede a força da relação linear entre duas variáveis contínuas. Acompanhado dele, utilizamos o **p-valor**, que avalia a probabilidade de encontrarmos essa correlação por acaso. Em análises clínicas, um p-valor $< 0.05$ sugere que a relação observada é estatisticamente significativa.

**🎯 Objetivo:** Importe a função `pearsonr` do `scipy.stats`. Calcule a correlação e o p-valor entre a `Idade` e o Risco (`y`). Repita o processo comparando o `Sodio_Serico` com o Risco.<br>

**🧠 Explicação lógica:**<br>
A função `pearsonr` da biblioteca Scipy recebe dois arrays (ou colunas de dataframe) e retorna sempre duas informações nessa exata ordem: o coeficiente de correlação estatística e o p-valor do teste de significância.

In [ ]:
"""
A função pearsonr retorna uma tupla: (correlação, p_valor)
"""
corr_idade, p_valor_idade = pearsonr(X['Idade'], y)
corr_sodio, p_valor_sodio = pearsonr(X['Sodio_Serico'], y)

print(f"Correlação Idade vs Risco: {corr_idade:.3f} (p-valor: {p_valor_idade:.4f})")
print(f"Correlação Sódio vs Risco: {corr_sodio:.3f} (p-valor: {p_valor_sodio:.4f})")

**Resposta da Pergunta:**
Apenas a variável **Idade** apresenta relação estatisticamente significativa com o Risco Cardiovascular, pois seu p-valor é consideravelmente menor que 0.05. O Sódio Sérico apresenta um p-valor alto ($> 0.05$), indicando que qualquer correlação observada na amostra é provavelmente obra do acaso (não possui utilidade clínica isolada para predizer o desfecho).

---

### ✅ **Resolução — Problema 1.3 - Teste de Hipótese para Grupos Clínicos** 🔴
Podemos estratificar a população e verificar se existe diferença real no risco entre grupos distintos utilizando o **Teste t de Student** para amostras independentes. Este teste compara as médias dos dois grupos para determinar se elas são significativamente diferentes.

**🎯 Objetivo:** Utilize a função `ttest_ind` do `scipy.stats` para realizar o teste comparando os escores de risco do grupo acima da média de idade (Grupo 1) com o grupo abaixo da média (Grupo 2).<br>

**🧠 Explicação lógica:**<br>
A função `ttest_ind` (T-test para amostras independentes) compara as médias de duas distribuições. Assim como a correlação, ela retorna a métrica do teste (Estatística T) e o p-valor.

In [ ]:
media_idade_dados = X['Idade'].mean()
risco_idade_alta = y[X['Idade'] > media_idade_dados]
risco_idade_baixa = y[X['Idade'] <= media_idade_dados]

"""
Passamos as duas parcelas da amostra separadas para a função de teste T
"""
estatistica_t, p_valor_t = ttest_ind(risco_idade_alta, risco_idade_baixa)

print(f"Estatística T: {estatistica_t:.2f}")
print(f"P-valor do Teste T: {p_valor_t:.5e}")

**Resposta da Pergunta:**
Eu responderia: "A diferença não é obra do acaso.". O p-valor obtido (geralmente com notação científica indicando um valor minúsculo, muito menor que 0.05) nos permite rejeitar a hipótese nula com altíssimo grau de confiança. Assim, provamos estatisticamente que os pacientes mais velhos pertencem a um grupo populacional de risco distinto dos pacientes mais novos.

---

> ## **Problema 2 - Construindo um Modelo Multivariado: Regressão Lasso** 🟡
No Problema 1, avaliamos os exames isoladamente. Contudo, em cenários reais, o risco é um fenômeno **multivariado**. Para considerar isso, construímos uma equação combinando todas as variáveis, atribuindo um **peso** (coeficiente $\beta$) a cada uma:

$$ Risco = (\beta_1 \times Idade) + (\beta_2 \times IMC) + ... + (\beta_{12} \times Potassio) + Erro $$

Se utilizarmos regressão linear comum, o sistema tentará atribuir um valor para todos os 12 pesos, deixando a equação extensa e sujeita a ruídos.

O método **Lasso** (Least Absolute Shrinkage and Selection Operator) ajusta esses pesos buscando minimizar a diferença entre o risco calculado e o risco real, porém introduzindo uma **penalidade** que reduz proporcionalmente a magnitude dos pesos. 

Matematicamente, a função de custo do modelo se torna:
$$ Custo = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} |\beta_j| $$

**Exemplo Prático:** Imagine um hospital que tem custo operacional por exame solicitado. O parâmetro $\alpha$ age como o "auditor". Para compensar a penalidade $\alpha$, o Lasso determina que variáveis pouco informativas tenham seu coeficiente ($\beta$) forçado para **zero**, selecionando ativamente apenas os exames relevantes.

### ✅ **Resolução — Problema 2.1 - Ajustando a Equação e Extraindo os Coeficientes** 🟡
Nós utilizaremos a classe `Lasso` do `sklearn` para encontrar os coeficientes. A intenção é observar graficamente o efeito da penalidade nos pesos calculados.

**🎯 Objetivo:** Preencha as lacunas no código `matplotlib` para gerar o gráfico de barras dos coeficientes ($\beta$) resultantes. <br>

**🧠 Explicação lógica:**<br>
Para criar um gráfico de barras (`ax.bar`), passamos as posições do eixo X (os índices correspondentes a cada variável do dataset) como o primeiro argumento e os valores das barras (os pesos matemáticos do Lasso) como o segundo argumento.

In [ ]:
calculador_betas = Lasso(alpha=10.0, random_state=42)
calculador_betas.fit(X, y)
valores_betas = calculador_betas.coef_

fig, ax = plt.subplots(figsize=(12, 5))

"""
O eixo x são os índices do tamanho da lista de colunas (np.arange(len(colunas))).
A altura de cada barra provém da matriz de coeficientes 'valores_betas'.
"""
ax.bar(np.arange(len(colunas)), valores_betas, color='royalblue')

ax.set_xticks(np.arange(len(colunas)))
ax.set_xticklabels(colunas, rotation=45, ha='right')
ax.set_ylabel("Valor do Coeficiente ($\beta$)")
ax.set_title("Perfil Multivariado: Coeficientes Calculados pelo Método Lasso")
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

**Resposta da Pergunta:**
Sim, a decisão do modelo está perfeitamente alinhada. No Problema 1.2, vimos que o Sódio Sérico produzia um p-valor não significativo (maior que 0.05). O Lasso, ao avaliar todas as variáveis em conjunto e buscar otimizar a equação penalizando variáveis de pouco impacto, convergiu para anular (zerar) exatamente variáveis com essa mesma característica.

---

### ✅ **Resolução — Problema 2.2 - Estimativas Clínicas (Interpolação com Scipy)** 🟢
Frequentemente não dispomos da equação matemática exata, operando com tabelas de referência que consolidam dados de populações. A interpolação linear permite inferir valores intermediários não tabelados.

**🎯 Objetivo:** Utilize `interp1d` do `scipy.interpolate` para criar uma função de interpolação baseada na tabela fornecida. Em seguida, calcule o risco estimado para um paciente de 53 anos.<br>

**🧠 Explicação lógica:**<br>
O objeto `interp1d` recebe o domínio `x` (idades conhecidas) e a imagem `y` (riscos) para modelar as retas que conectam esses pontos discretos. Feito isso, ele se torna uma função mapeável para estimar valores desconhecidos.

In [ ]:
idades_conhecidas = np.array([30, 40, 50, 60, 70])
riscos_medios = np.array([12.5, 28.0, 45.2, 68.1, 95.0])

"""
Passo 1: Instanciar a função baseada nos dados experimentais.
Passo 2: Injetar o novo valor (53) nessa nova função contínua.
"""
funcao_interp = interp1d(idades_conhecidas, riscos_medios)
risco_estimado_53 = funcao_interp(53)

print(f"Risco interpolado para paciente de 53 anos: {risco_estimado_53:.2f}")

---

> ## **Problema 3 - Avaliando a Qualidade e Significância do Modelo** 🟡

Construir uma fórmula ou aplicar um método preditivo não encerra a análise; é necessário atestar a confiabilidade matemática desse modelo. 

Para isso, utilizamos métricas que quantificam a qualidade do ajuste e o nível de erro:

* **$R^2$ (Coeficiente de Determinação):** Mede a proporção da variabilidade do Escore de Risco que é explicada pela equação construída. O valor varia de 0 a 1, indicando o quão bem o modelo se aproxima dos dados originais.
  
  $$R^2 = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}$$
  *(Onde $y_i$ é o risco real observado, $\hat{y}_i$ é o risco calculado pela equação, e $\bar{y}$ é a média dos riscos reais).* 

* **MSE (Erro Quadrático Médio):** Representa a média das diferenças ao quadrado entre os valores reais e os calculados pela equação. Valores menores indicam maior precisão.

  $$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

### ✅ **Resolução — Problema 3.1 - Métricas de Erro e o Relatório Statsmodels** 🟡
Enquanto bibliotecas mais recentes geram previsões eficientes, abordagens baseadas em estatística clássica — como o pacote `statsmodels` — fornecem um sumário detalhado que descreve a significância estatística (p-valor) para cada coeficiente da regressão.

**🎯 Objetivo:** Calcule as métricas $R^2$ e MSE a partir das estimativas do modelo Lasso. Logo após, utilize o `statsmodels` para realizar a regressão comum (OLS) e exiba o relatório estatístico (*summary*).<br>

**🧠 Explicação lógica:**<br>
Tanto a função `r2_score` quanto `mean_squared_error` comparam os valores originais reais (`y`) contra os valores previstos/calculados pelo modelo (`riscos_calculados`). O `statsmodels` exige que seja gerado um `summary()` do objeto ajustado para compor o balanço das métricas.

In [ ]:
riscos_calculados = calculador_betas.predict(X)

"""
Cálculo das métricas de regressão importadas do Scikit-Learn
"""
r2_equacao = r2_score(y, riscos_calculados)
mse_equacao = mean_squared_error(y, riscos_calculados)

print(f"Desempenho da Equação -> R²: {r2_equacao:.3f} | MSE: {mse_equacao:.2f}\n")
print("="*70)

# Relatório estatístico detalhado via Statsmodels
X_com_constante = sm.add_constant(X)
modelo_estatistico = sm.OLS(y, X_com_constante).fit()

"""
Apresentação do summary formatado nativo do objeto statsmodels
"""
print(modelo_estatistico.summary())

**Resposta da Pergunta:**
Sim, a estatística clássica confirma e justifica. Quando analisamos a tabela gerada pelo `statsmodels`, variáveis como o Colesterol HDL e o Sódio Sérico exibem valores elevados na coluna `P>|t|` (p-valor $> 0.05$). Na estatística clássica, a regressão OLS identifica que esses exames não têm peso validado clinicamente frente a variação conjunta. O Lasso confirma essa lógica de forma algorítmica reduzindo o peso delas a zero absoluto em sua execução.

---

### ✅ **Resolução — Problema 3.2 - Desmembrando os Efeitos (Direção e Confiança)** 🟢
O relatório estatístico vai muito além de dizer se o modelo "funcionou ou não". Ele detalha a contribuição individual de cada marcador clínico.

**🎯 Objetivo:** Com base na tabela (summary) gerada no Problema 3.1, observe atentamente as colunas `coef` (coeficiente do impacto numérico), `P>|t|` (p-valor) e `[0.025  0.975]` (Intervalo de Confiança a 95%) e responda as perguntas abaixo.

**🧠 Explicação lógica e Respostas:**
*(Nota: Os nomes exatos das variáveis de maior impacto variam levemente porque o banco de dados simulou os impactos de maneira aleatória, mas a lógica de leitura da tabela abaixo é universal).* 

1. **Menor p-valor:** Devemos olhar a coluna `P>|t|`. As variáveis que o simulador definiu secretamente como importantes terão o p-valor igual a `0.000`. Estas são estatisticamente significativas.
2. **Maior influência positiva:** Devemos buscar o maior valor numérico positivo na coluna `coef`. Um coeficiente alto e positivo significa que, conforme o resultado desse exame aumenta, o Risco Cardiovascular do paciente sobe drasticamente.
3. **Coeficiente numérico negativo:** Sim, os algoritmos podem gerar coeficientes negativos na coluna `coef` (ex: na vida real, o HDL alto costuma ter `coef` negativo). Clinicamente, isso representa um **fator de proteção**: o aumento dessa variável *diminui* o Escore de Risco global do paciente.
4. **Intervalo de confiança cruzando zero:** Sim, muitas das variáveis que são apenas "ruído" terão intervalos na coluna `[0.025  0.975]` que vão de um valor negativo até um positivo (ex: `-1.50` a `2.10`). Se o zero está contido nesse intervalo de confiança, significa que o modelo não tem certeza estatística se a variável aumenta (efeito positivo) ou diminui (efeito negativo) o risco. Isso ratifica de outra forma a falta de significância (p-valor $> 0.05$).

---

### ✅ **Resolução — Problema 3.3 - As Armadilhas das Métricas Globais ($R^2$)** 🟡
Frequentemente, em pesquisas médicas, observamos o valor do $R^2$ sendo utilizado como a prova definitiva de que um modelo é perfeito.

**🎯 Objetivo:** Utilize seu conhecimento crítico e os resultados das etapas anteriores para responder às questões conceituais.

**🧠 Explicação lógica e Respostas:**

1. **$R^2$ elevado com p-valores altos:** Sim, isso é totalmente possível. A matemática por trás do $R^2$ tradicional faz com que ele **sempre aumente** (ou se mantenha igual) quando adicionamos novas variáveis à equação, mesmo que sejam puro ruído matemático. Assim, se você adicionar 50 variáveis inúteis a um modelo que contém 2 variáveis excelentes, o modelo terá um $R^2$ global artificialmente elevado (pela soma), mas a tabela *summary* revelará 50 p-valores altos indicando que a grande maioria dos exames inseridos não tem utilidade nenhuma.
2. **O $R^2$ sozinho é suficiente?** Não, de forma alguma. Um modelo com alto $R^2$ pode estar sofrendo de *overfitting* (sobreajuste), decorando os pacientes da base de dados local, mas falhando miseravelmente ao tentar prever os riscos de pacientes reais de outros hospitais. Além disso, o $R^2$ não avalia o tamanho do erro absoluto na mesma unidade da doença (por isso o MSE é fundamental), nem atesta se a correlação tem coerência fisiológica. Ele é apenas uma das dezenas de métricas necessárias para homologar um protocolo clínico.

---